# Track A: Feature Engineering (Bank Transactions)
This notebook transforms the raw transaction log into a customer-level behavioral matrix using RFM (Recency, Frequency, Monetary) aggregations and demographic engineering.


In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.preprocessing import StandardScaler
import os

pd.set_option('display.max_columns', None)


## 1. Load Data


In [2]:
df = pd.read_csv('../data/bank_transactions.csv')
print(f'Raw data shape: {df.shape}')
df.dropna(inplace=True)
print(f'Shape after dropping NAs: {df.shape}')


Raw data shape: (1048567, 9)
Shape after dropping NAs: (1041614, 9)


## 2. Demographic Engineering


In [3]:
# Extract Age with Century Pivot
try:
    df['DOB'] = pd.to_datetime(df['CustomerDOB'], format='%d/%m/%y', errors='coerce')
    # Pandas parses e.g., '73' as 2073. If year > 2005, it belongs to the 1900s.
    df.loc[df['DOB'].dt.year > 2005, 'DOB'] -= pd.DateOffset(years=100)
    # Calculate Age relative to 2016 (when transactions occurred)
    df['Age'] = 2016 - df['DOB'].dt.year
    df['Age'] = np.where((df['Age'] < 18) | (df['Age'] > 100), np.nan, df['Age'])
except Exception as e:
    pass

# One-Hot Encode Gender
df = pd.get_dummies(df, columns=['CustGender'], drop_first=True, dtype=int)

# Parse Transaction Date
df['TransactionDate'] = pd.to_datetime(df['TransactionDate'], format='%d/%m/%y', errors='coerce')

## 3. Customer-Level Aggregation (RFM)


In [4]:
max_date = df['TransactionDate'].max()

# Calculate Spend-to-Balance Ratio (add 1 to avoid div by zero)
df['Spend_Balance_Ratio'] = df['TransactionAmount (INR)'] / (df['CustAccountBalance'] + 1)

# Aggregate to Customer Level
customer_df = df.groupby('CustomerID').agg(
    Age=('Age', 'first'),
    Gender_M=('CustGender_M', 'first') if 'CustGender_M' in df.columns else ('Age', 'first'),
    AccountBalance=('CustAccountBalance', 'mean'),
    
    # Recency
    Last_Transaction=('TransactionDate', 'max'),
    
    # Frequency
    Transaction_Count=('TransactionID', 'count'),
    
    # Monetary
    Avg_Transaction_Amt=('TransactionAmount (INR)', 'mean'),
    Total_Spend=('TransactionAmount (INR)', 'sum'),
    
    # Behavioral
    Avg_Spend_Ratio=('Spend_Balance_Ratio', 'mean')
).reset_index()

if 'CustGender_M' not in df.columns:
    customer_df.drop(columns=['Gender_M'], inplace=True, errors='ignore')

# Calculate Days Since Last Transaction (Recency)
customer_df['Recency_Days'] = (max_date - customer_df['Last_Transaction']).dt.days
customer_df.drop(columns=['Last_Transaction'], inplace=True)

customer_df.dropna(inplace=True)
print(f'Customer-level dataframe shape: {customer_df.shape}')
display(customer_df.head())


Customer-level dataframe shape: (837951, 9)


,CustomerID,Age,Gender_M,AccountBalance,Transaction_Count,Avg_Transaction_Amt,Total_Spend,Avg_Spend_Ratio,Recency_Days
0,C1010011,24.0,0,76340.635,2,2553.0,5106.0,0.074554,25
1,C1010012,22.0,1,24204.490,1,1499.0,1499.0,0.061928,68
2,C1010014,24.0,0,100112.950,2,727.5,1455.0,0.016471,75
3,C1010018,26.0,0,496.180,1,30.0,30.0,0.060340,36
4,C1010024,51.0,1,87058.650,1,5000.0,5000.0,0.057432,64


## 4. Normalization and Saving
We apply Log1p to monetary columns and StandardScaler everywhere else.


In [5]:
features = ['Age', 'AccountBalance', 'Transaction_Count', 'Avg_Transaction_Amt', 'Total_Spend', 'Avg_Spend_Ratio', 'Recency_Days']
features = [f for f in features if f in customer_df.columns]

# Log transformations for heavily skewed monetary features
for col in ['AccountBalance', 'Avg_Transaction_Amt', 'Total_Spend']:
    customer_df[col] = np.log1p(customer_df[col])

scaler = StandardScaler()
customer_df[features] = scaler.fit_transform(customer_df[features])

# Save to CSV to bypass PyArrow bug
os.makedirs('../data/processed', exist_ok=True)
customer_df.to_csv('../data/processed/track_a_features.csv', index=False)
print('Track A features successfully saved to data/processed/track_a_features.csv')


Track A features successfully saved to data/processed/track_a_features.csv
